# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/supriya-006/FlyRank_Assignment/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I am using a `RandomForestClassifier` for the learned model because the target is a binary page-health label (`is_declining_label`) and the signal is not purely linear: traffic, position, freshness, and content depth interact with each other. A tree ensemble captures those nonlinear interactions while still giving interpretable feature importance. I compare it directly to the transparent Week-4 rule baseline on the same client-held-out split and the same ranking metric, `precision@50`.


In [2]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, start.parent, start.parent.parent]:
        if candidate == candidate.parent:
            continue
        if (candidate / 'data').exists() and (candidate / 'scripts').exists():
            return candidate
        if (candidate / 'work').exists() and (candidate / 'data').exists():
            return candidate
    return start.resolve()


ROOT = find_repo_root(Path.cwd().resolve())
if (ROOT / 'scripts').exists():
    sys.path.insert(0, str(ROOT / 'scripts'))
elif (ROOT.parent / 'scripts').exists():
    sys.path.insert(0, str(ROOT.parent / 'scripts'))

print('Repo root:', ROOT)
print('Script utilities available:', (ROOT / 'scripts').exists() or (ROOT.parent / 'scripts').exists())


Repo root: /home/supriya-devkota/Desktop/FlyRank_Assignment
Script utilities available: True


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

The honest split is a client-holdout split. The content rows are not independent: pages from the same client share a traffic profile, ranking history, and editorial workflow. A row-level random split would leak client-level patterns into training and evaluation. I therefore hold out an entire client group for testing, which makes the comparison more realistic and prevents over-optimistic performance from portfolio effects.


In [3]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

FEATURE_PATH = ROOT / 'data' / 'processed' / 'refresh_feature_vector.csv'
BASELINE_PATH = ROOT / 'data' / 'processed' / 'baseline_refresh_queue.csv'

print('Looking for features at:', FEATURE_PATH)
print('Looking for baseline at:', BASELINE_PATH)

if not FEATURE_PATH.exists():
    raise FileNotFoundError(f"Missing feature vector: {FEATURE_PATH}")
if not BASELINE_PATH.exists():
    raise FileNotFoundError(f"Missing baseline queue: {BASELINE_PATH}")

df = pd.read_csv(FEATURE_PATH)
baseline_df = pd.read_csv(BASELINE_PATH)

print('Rows:', len(df))
print('Declining rate:', round(df['is_declining_label'].mean(), 3))

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print('Train rows:', len(train_df), 'Test rows:', len(test_df))
print('Train positive rate:', round(train_df['is_declining_label'].mean(), 3))
print('Test positive rate:', round(test_df['is_declining_label'].mean(), 3))
print('Test clients:', test_df['client_id'].nunique())


Looking for features at: /home/supriya-devkota/Desktop/FlyRank_Assignment/data/processed/refresh_feature_vector.csv
Looking for baseline at: /home/supriya-devkota/Desktop/FlyRank_Assignment/data/processed/baseline_refresh_queue.csv
Rows: 30000
Declining rate: 0.542
Train rows: 23837 Test rows: 6163
Train positive rate: 0.55
Test positive rate: 0.511
Test clients: 7


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The notebook uses the same feature set and test split as the baseline queue, and ranks content by predicted declining probability. The comparison metric is `precision@50` because the decision is a top-K review action: the team will only act on the highest-scoring pages.


In [4]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from ml_utils import MODEL_CATEGORICAL_FEATURES, MODEL_NUMERIC_FEATURES, precision_at_k

numeric_features = [c for c in MODEL_NUMERIC_FEATURES if c in df.columns]
categorical_features = [c for c in MODEL_CATEGORICAL_FEATURES if c in df.columns]

X_train = train_df[numeric_features + categorical_features].copy()
X_test = test_df[numeric_features + categorical_features].copy()
y_train = train_df['is_declining_label'].astype(int)
y_test = test_df['is_declining_label'].astype(int)

preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]),
            numeric_features,
        ),
        (
            'cat',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore')),
            ]),
            categorical_features,
        ),
    ]
)

models = {
    'Logistic Regression': Pipeline([
        ('preprocess', preprocessor),
        ('model', LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)),
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=25,
        class_weight='balanced_subsample',
        random_state=42,
    ),
}

def summary_frame(y_true, scores, label):
    pred = (scores >= 0.5).astype(int)
    return {
        'method': label,
        'accuracy': round(accuracy_score(y_true, pred), 4),
        'precision': round(precision_score(y_true, pred, zero_division=0), 4),
        'recall': round(recall_score(y_true, pred, zero_division=0), 4),
        'f1': round(f1_score(y_true, pred, zero_division=0), 4),
        'roc_auc': round(roc_auc_score(y_true, scores), 4),
        'precision_at_20': round(precision_at_k(y_true, scores, 20), 4),
        'precision_at_50': round(precision_at_k(y_true, scores, 50), 4),
        'precision_at_100': round(precision_at_k(y_true, scores, 100), 4),
    }

baseline_lookup = baseline_df.set_index('content_id')['baseline_refresh_score']
baseline_scores = test_df['content_id'].map(baseline_lookup).fillna(0).to_numpy()
baseline_summary = summary_frame(y_test.to_numpy(), baseline_scores, 'Baseline')

model_results = []
for model_name, model in models.items():
    if model_name == 'Logistic Regression':
        fitted_model = model.fit(X_train, y_train)
        scores = fitted_model.predict_proba(X_test)[:, 1]
    else:
        fitted_model = Pipeline([
            ('preprocess', preprocessor),
            ('model', model),
        ])
        fitted_model = fitted_model.fit(X_train, y_train)
        scores = fitted_model.predict_proba(X_test)[:, 1]
    model_results.append(summary_frame(y_test.to_numpy(), scores, model_name))

comparison = pd.DataFrame([baseline_summary, *model_results])
comparison = comparison.sort_values('precision_at_50', ascending=False).reset_index(drop=True)

output_dir = ROOT / 'work' / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)
comparison.to_csv(output_dir / 'w05_model_comparison.csv', index=False)

print('Model-vs-baseline comparison:')
print(comparison.to_string(index=False))
comparison


Model-vs-baseline comparison:
             method  accuracy  precision  recall     f1  roc_auc  precision_at_20  precision_at_50  precision_at_100
Logistic Regression    0.5832     0.5861  0.6269 0.6058   0.6158             0.70             0.72              0.71
      Random Forest    0.5814     0.5889  0.5983 0.5936   0.6090             0.55             0.58              0.58
           Baseline    0.4676     0.4610  0.2477 0.3222   0.4979             0.40             0.32              0.31


,method,accuracy,precision,recall,f1,roc_auc,precision_at_20,precision_at_50,precision_at_100
0,Logistic Regression,0.5832,0.5861,0.6269,0.6058,0.6158,0.70,0.72,0.71
1,Random Forest,0.5814,0.5889,0.5983,0.5936,0.6090,0.55,0.58,0.58
2,Baseline,0.4676,0.4610,0.2477,0.3222,0.4979,0.40,0.32,0.31


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The learned model improves the top-of-queue review quality, but the remaining errors are concentrated in borderline pages: high impression pages that are still not clearly declining, and pages with strong rankings but weak engagement signals. Those are exactly the cases where one more click, stale metadata, or a seasonality effect can flip the label.


In [5]:
rf_model = Pipeline([
    ('preprocess', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=25,
        class_weight='balanced_subsample',
        random_state=42,
    )),
])
rf_model.fit(X_train, y_train)
rf_scores = rf_model.predict_proba(X_test)[:, 1]
rf_pred = (rf_scores >= 0.5).astype(int)

error_df = test_df[['content_id', 'client_id', 'is_declining_label', 'avg_position', 'days_since_last_update', 'ctr', 'impressions_90d', 'word_count']].copy()
error_df['pred'] = rf_pred
error_df['probability'] = rf_scores
error_df['error'] = (error_df['is_declining_label'] != error_df['pred']).astype(int)

false_positives = error_df[(error_df['is_declining_label'] == 0) & (error_df['pred'] == 1)].sort_values('probability', ascending=False)
false_negatives = error_df[(error_df['is_declining_label'] == 1) & (error_df['pred'] == 0)].sort_values('probability', ascending=True)

print('False positives (top 5):')
print(false_positives[['content_id', 'client_id', 'probability', 'avg_position', 'days_since_last_update', 'ctr', 'impressions_90d']].head(5).to_string(index=False))
print('False negatives (top 5):')
print(false_negatives[['content_id', 'client_id', 'probability', 'avg_position', 'days_since_last_update', 'ctr', 'impressions_90d']].head(5).to_string(index=False))

feature_names = rf_model.named_steps['preprocess'].get_feature_names_out()
feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_model.named_steps['model'].feature_importances_,
}).sort_values('importance', ascending=False).head(10)

print('Top random-forest feature importances:')
print(feature_importance.to_string(index=False))

fp_summary = false_positives[['avg_position', 'days_since_last_update', 'ctr', 'impressions_90d']].describe().T
fn_summary = false_negatives[['avg_position', 'days_since_last_update', 'ctr', 'impressions_90d']].describe().T
print('False positive summary:')
print(fp_summary.to_string())
print('False negative summary:')
print(fn_summary.to_string())


False positives (top 5):
          content_id         client_id  probability  avg_position  days_since_last_update  ctr  impressions_90d
content_2ba626fea4d6 client_8527a891e2     0.857676           7.2                     104 0.00              360
content_35d63627bf3e client_8527a891e2     0.849607          32.6                     103 0.00             1525
content_1d0963b56227 client_4e07408562     0.846989          39.0                     104 0.09             3445
content_0b47dae0c7f9 client_8527a891e2     0.846249          23.1                     103 0.00             1191
content_846bb4dd8b44 client_8527a891e2     0.842497          17.6                     104 0.11              870
False negatives (top 5):
          content_id         client_id  probability  avg_position  days_since_last_update  ctr  impressions_90d
content_16f38acf0f26 client_e629fa6598     0.106755          50.0                      20  0.0                2
content_7bc32bc1df59 client_8527a891e2     0.128450   

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
